In [1]:
## Required libraries
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import time

In [2]:
# Enter the base data location
base_url = 'C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\Accor Dep JAN_firstJan.csv'
base_df=pd.read_csv(base_url, encoding = "ISO-8859-1", parse_dates=['Arrival_Date','Departure_Date'])

In [3]:
# Required functions

def preprocess(x):
    
    x["Arrival"]= pd.to_datetime(x["Arrival"]) 
    x["Departure"]= pd.to_datetime(x["Departure"]) 
    x["Last Name"] = x["Last Name"].astype(str)
    result = pd.merge(left = base_df, 
                  right = x, 
                  left_on = ['Arrival_Date', 'Departure_Date'], 
                  right_on =['Arrival', 'Departure'], 
                  how = 'right')
    matched_df = result[result.Reservation_No.notnull() == True]
    unmatched_df = result[result.Reservation_No.notnull() == False]
    
    base_test = matched_df["Client_Guestname_1"].str.strip()
    base_test = base_test.str.replace(',','')
    base_test = base_test.str.upper()
    
    ground_truth = matched_df["Last Name"].values.tolist()
    raw_test = matched_df["Last Name"].str.strip()
    raw_test = raw_test.str.upper()
    # list conversion for fuzzy matching
    raw_list = raw_test.values.tolist()
    base_list = base_test.values.tolist()
    
    # Fuzzy matching
    possibilities = []
    for string in raw_list:
        #print(string)
        possibility = process.extractOne(string, base_list, scorer=fuzz.token_sort_ratio)
        possibilities.append(possibility)
    temp_df = pd.DataFrame(possibilities, columns = ['Match_list', 'Match_score'])
    temp_df['Actual_string'] = ground_truth
    temp_df['Actual_string'] = temp_df['Actual_string'].astype(str)
    temp_df.columns = ['Match_list', 'Match_score','Actual_string']
    interm_2 = pd.merge(left = matched_df, right = temp_df, left_on = 'Last Name', right_on = 'Actual_string' , how = 'left')
    interm_3 = interm_2.append(unmatched_df) 
    threshold = 70
    interm_3['Match_status'] = interm_3['Match_score'].apply(lambda x: 'Y' if x > threshold else 'N')
    interm_3['Confirmation'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Reservation_No'], interm_3['Confirmation'])
    interm_3['Last Name'] = np.where(interm_3['Match_status'] == 'Y', interm_3['Client_Guestname_1'], interm_3['Last Name'])
    raw_col = list(x.columns)
    raw_col.append("Match_status")
    final_raw = interm_3[raw_col]
    final_raw = final_raw.drop_duplicates()
    ## Result raw file location
    final_raw.to_csv("C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\result_raw1.csv",mode="a",sep = ',',header=True,index=False)

In [4]:
## Execution

# Enter the raw data
raw_url = "C:\\Users\\svi02\\Documents\\Python Scripts\\Matching\\11-HRS_periodend_14_03_2020.csv"

reader = pd.read_csv(raw_url, chunksize=1000, encoding = "ISO-8859-1", parse_dates=['Arrival','Departure','Pay Date'])
start_time = time.time()
#[preprocess(r) for r in reader]
for r in reader:
    preprocess(r)
    print(r.shape)
    
end_time = time.time()
print(end_time - start_time)
print('SUCCESS')

C:\Anaconda\lib\site-packages\pandas\core\frame.py:7138: FutureWarning: Sorting because non-concatenation axis is not aligned. A future version
of pandas will change to not sort by default.

To accept the future behavior, pass 'sort=False'.

To retain the current behavior and silence the warning, pass 'sort=True'.

  sort=sort,


(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(1000, 44)
(168, 44)
146.74521398544312
SUCCESS
